# Phase 6.3.1 — Offline Decision Protocol Replay
Replay all 18 frozen Phase 6.3B responses. No API keys, model calls, embedding downloads, new retrieval, manual uploads or library changes. The original run and persistent attempt cache are read-only. Recommendation is rank-first; supported selection remains nullable. Exact citations are not proof of semantic applicability. This is development-only evaluation.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib, shutil, json
REPO_URL = 'https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH = 'simplified-rag-v3'
ROOT = Path('/content/tfl-simplified-rag-v3')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[retrieval,test]'],check=True)
SRC = ROOT/'src'
sys.path.insert(0,str(SRC))
for name in list(sys.modules):
    if name=='tfl_agent' or name.startswith('tfl_agent.'):
        del sys.modules[name]
for p in SRC.joinpath('tfl_agent').rglob('__pycache__'):
    shutil.rmtree(p)
importlib.invalidate_caches()
import tfl_agent
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC.resolve())
print('Commit:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
print('Agent:',tfl_agent.__file__)
subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests/test_phase6_decision_replay.py'),str(ROOT/'tests/test_phase6_llm_pilot.py')],check=True)

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from tfl_agent.agent.phase6_decision_replay import ARCHIVE_NAME, ARCHIVE_SHA, run_replay
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo')
ARCHIVE = BASE/'Phase63_LLM_Pilot_Outputs'/ARCHIVE_NAME
if not ARCHIVE.is_file():
    raise FileNotFoundError(f'Expected exact historical run: {ARCHIVE}. Available: {[p.name for p in ARCHIVE.parent.glob("*.zip")]}')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
print('Frozen archive:', ARCHIVE)
print('Expected SHA256:', ARCHIVE_SHA)
print('New LLM calls authorized: 0')

In [ ]:
import pandas as pd
from IPython.display import display
OUT = Path('/content/Phase631_Decision_Replay')/RUN_ID
report, rows = run_replay(ARCHIVE, OUT)
assert report['new_llm_calls']==0 and len(rows)==18
assert not any(r['generation_authorized'] or r['clinical_confirmation'] for r in rows)
metrics = []
for arm, m in report['metrics'].items():
    metrics.append({'arm':arm,'n':m['n'],'raw_top1':m['raw_top1'],
                    'recall_at_3':m['raw_recall_at_3'],'MRR':m['raw_mrr'],
                    'old_validator_pass':m['original_strict_pass'],
                    'new_protocol_pass':m['revised_protocol_pass'],
                    'repaired':m['protocol_repairs'],
                    'original_correct_selected':m['original_strict_selection']['correct'],
                    'recommended_correct':m['revised_recommendation']['correct'],
                    'supported_correct':m['revised_supported_selection']['correct'],
                    'supported_count':m['revised_supported_selection']['selected']})
display(pd.DataFrame(metrics))
df = pd.DataFrame(rows)
display(df[['study_id','entry_id','arm','expected','expected_rank','original_record_status',
            'original_selection_status','recommended_template','supported_template',
            'protocol_repaired','issues']])
print('Recorded historical pilot usage:',report['historical_recorded_usage'])
print('New LLM calls:',report['new_llm_calls'])
print('Protocol repair is not an increase in supported selection or ranking accuracy.')

In [ ]:
DEST = BASE/'Phase631_Decision_Replay_Outputs'
DEST.mkdir(parents=True,exist_ok=True)
saved = Path(shutil.make_archive(str(DEST/RUN_ID),'zip',root_dir=OUT))
print('Saved:',saved)
print('Files:', sorted(p.name for p in OUT.iterdir()))
print('Please share this ZIP for Phase 6.3.2 evidence applicability review. It contains confidential SAP quotations; share only if authorized.')

## Interpretation
- Raw top-1, recall@3 and MRR are computed over all six cases per arm, including previous protocol failures.
- Original strict selection counts only responses accepted by the original validator. Revised recommendation is a rank-first diagnostic, even when support is missing or the model explicitly abstains. It is not an accepted clinical structure.
- Revised supported selection requires valid candidate-only status, IDs, assessments, citations and preset ownership. An uncertain non-null proposal is repaired into a separate recommendation, not a supported selection.
- Citation_Audit.csv preserves exact quotes and marks applicability unreviewed. Phase 6.3.2 will review target specificity, other-endpoint and broad topic evidence. Do not rerun the 18-call pilot or start a new prompt pilot.